# Missing Parish Analysis for Louisiana OSHA Accident Data

In [2]:
import pandas as pd
import numpy as np

file_path = "../data/processed/LouisianaAccidentData_with_parish.csv"

df = pd.read_csv(file_path)
df.shape

(2319, 28)

## Verifying that the processed file contains the expected geography and output fields:
- `city`
- `Zip`
- `Latitude`
- `Longitude`
- `parish`


In [3]:
df.columns.tolist()

['ID',
 'UPA',
 'EventDate',
 'Employer',
 'Address1',
 'Address2',
 'city',
 'state',
 'Zip',
 'Latitude',
 'Longitude',
 'Primary_NAICS',
 'Hospitalized',
 'Amputation',
 'Loss_Of_Eye',
 'Inspection',
 'FinalNarrative',
 'Nature',
 'NatureTitle',
 'Part_of_Body',
 'Part_of_Body_Title',
 'Event',
 'EventTitle',
 'Source',
 'sourceTitle',
 'Secondary_Source',
 'Secondary_Source_Title',
 'parish']

## Counting missing parish values

In [4]:
missing_parish = df[df["parish"].isna()].copy()

n_total = len(df)
n_missing = len(missing_parish)
pct_missing = round((n_missing / n_total) * 100, 4)

print(f"Total records: {n_total:,}")
print(f"Missing parish records: {n_missing:,}")
print(f"Percent missing: {pct_missing}%")

Total records: 2,319
Missing parish records: 16
Percent missing: 0.69%


## Preparing the unmatched records for inspection

We standardize city and ZIP so the rows are easier to review and later patch if needed.

In [5]:
missing_parish["EventDate"] = pd.to_datetime(missing_parish["EventDate"], errors="coerce")
missing_parish["city_clean"] = missing_parish["city"].astype(str).str.strip().str.upper()
missing_parish["zip5"] = missing_parish["Zip"].astype(str).str.extract(r"(\d{5})", expand=False)

cols_to_show = [
    "ID", "EventDate", "Employer", "city_clean", "zip5",
    "Latitude", "Longitude", "parish"
]

missing_parish[cols_to_show].sort_values(["city_clean", "zip5"]).reset_index(drop=True)

,ID,EventDate,Employer,city_clean,zip5,Latitude,Longitude,parish
0,2022065383,2022-06-21,Estes Express Lines #85,BATON ROUGE,70806,39.04,-76.49,NaN
1,2018077574,2018-07-26,Jeffrey Jackson,CHALMETTE,70043,40.62,-81.78,NaN
2,2019077381,2019-07-19,Waste Management of Lake Charles,DEQUINCY,70633,28.23,-81.65,NaN
3,2018087835,2018-08-01,Helmerich & Payne IDC,ELM GROVE,71051,37.09,-95.71,NaN
4,20191111804,2019-11-13,Cox Communications Louisiana,LAFAYETTE,70506,41.18,-87.47,NaN
5,2017010202,2017-01-09,"Waste Management of Louisiana, LLC",LAKE CHARLES,70605,36.73,-75.94,NaN
6,2019099926,2019-09-23,Sunoco Energy Services,LONGSTREET,71050,41.08,-79.16,NaN
7,2024065349,2024-06-17,Schlumberger Technology Corporation,MANSFIELD,71052,44.01,-102.23,NaN
8,2024043063,2024-04-08,JF Petro Group,METAIRIE,70001,41.97,-70.70,NaN
9,2024043202,2024-04-11,"H&O Investments, LLC",METAIRIE,70001,39.13,-76.69,NaN


## Diagnosing the reason for the missing parish values

A valid Louisiana coordinate should usually fall roughly within:
- latitude: about **28 to 34**
- longitude: about **-95 to -88**

This is not a formal boundary test but a quick diagnostic to identify obviously misplaced geocodes.

In [10]:
missing_parish["coord_missing"] = missing_parish["Latitude"].isna() | missing_parish["Longitude"].isna()
missing_parish["coord_outside_louisiana_range"] = ~(
    missing_parish["Latitude"].between(28, 34, inclusive="both") &
    missing_parish["Longitude"].between(-95, -88, inclusive="both")
) & (missing_parish["Latitude"].notna() | missing_parish["Longitude"].notna())

diagnostic_summary = pd.DataFrame({
    "metric": [
        "Missing latitude and/or longitude",
        "Outside rough Louisiana coordinate range",
        "Total missing parish rows"
    ],
    "count": [
        int(missing_parish["coord_missing"].sum()),
        int(missing_parish["coord_outside_louisiana_range"].sum()),
        len(missing_parish)
    ]
})

diagnostic_summary

,metric,count
0,Missing latitude and/or longitude,1
1,Outside rough Louisiana coordinate range,15
2,Total missing parish rows,16


## Reviewing the problematic geography

The rows below show Louisiana cities and ZIP codes paired with coordinates from other U.S. states or even outside the U.S. That indicates **source geocoding error**, not a parish boundary mismatch.

In [11]:
missing_parish[
    ["city_clean", "zip5", "Latitude", "Longitude", "coord_missing", "coord_outside_louisiana_range"]
].sort_values(["city_clean", "zip5"]).reset_index(drop=True)

,city_clean,zip5,Latitude,Longitude,coord_missing,coord_outside_louisiana_range
0,BATON ROUGE,70806,39.04,-76.49,False,True
1,CHALMETTE,70043,40.62,-81.78,False,True
2,DEQUINCY,70633,28.23,-81.65,False,True
3,ELM GROVE,71051,37.09,-95.71,False,True
4,LAFAYETTE,70506,41.18,-87.47,False,True
5,LAKE CHARLES,70605,36.73,-75.94,False,True
6,LONGSTREET,71050,41.08,-79.16,False,True
7,MANSFIELD,71052,44.01,-102.23,False,True
8,METAIRIE,70001,41.97,-70.70,False,True
9,METAIRIE,70001,39.13,-76.69,False,True


## Creating a fallback parish mapping for the affected rows

Because the number of unresolved rows is very small, we used the city and ZIP columns instead to label the parishes.

- **Primary method:** spatial join using coordinates and Louisiana parish polygons
- **Fallback method:** utilizing city and ZIP to parish mapping for the small number of geocoding failures


In [14]:
fallback_map = {
    ("METAIRIE", "70001"): "Jefferson",
    ("MANSFIELD", "71052"): "De Soto",
    ("DEQUINCY", "70633"): "Calcasieu",
    ("WALKER", "70785"): "Livingston",
    ("LONGSTREET", "71050"): "De Soto",
    ("LAFAYETTE", "70506"): "Lafayette",
    ("LAKE CHARLES", "70605"): "Calcasieu",
    ("PONCHATOULA", "70454"): "Tangipahoa",
    ("NEW ORLEANS", "70118"): "Orleans",
    ("STONEWALL", "71078"): "De Soto",
    ("CHALMETTE", "70043"): "St. Bernard",
    ("ELM GROVE", "71051"): "Bossier",
    ("BATON ROUGE", "70806"): "East Baton Rouge",
    ("METAIRIE", "70003"): "Jefferson",
}

missing_parish["fallback_parish"] = missing_parish.apply(
    lambda row: fallback_map.get((row["city_clean"], row["zip5"])),
    axis=1
)

missing_parish[
    ["city_clean", "zip5", "Latitude", "Longitude", "fallback_parish"]
].sort_values(["city_clean", "zip5"]).reset_index(drop=True)

,city_clean,zip5,Latitude,Longitude,fallback_parish
0,BATON ROUGE,70806,39.04,-76.49,East Baton Rouge
1,CHALMETTE,70043,40.62,-81.78,St. Bernard
2,DEQUINCY,70633,28.23,-81.65,Calcasieu
3,ELM GROVE,71051,37.09,-95.71,Bossier
4,LAFAYETTE,70506,41.18,-87.47,Lafayette
5,LAKE CHARLES,70605,36.73,-75.94,Calcasieu
6,LONGSTREET,71050,41.08,-79.16,De Soto
7,MANSFIELD,71052,44.01,-102.23,De Soto
8,METAIRIE,70001,41.97,-70.70,Jefferson
9,METAIRIE,70001,39.13,-76.69,Jefferson


## Estimating how many rows the fallback resolves

In [15]:
resolved_by_fallback = missing_parish["fallback_parish"].notna().sum()
remaining_after_fallback = missing_parish["fallback_parish"].isna().sum()

print(f"Rows resolved by fallback: {resolved_by_fallback}")
print(f"Rows still unresolved after fallback: {remaining_after_fallback}")

Rows resolved by fallback: 16
Rows still unresolved after fallback: 0
